# Amazon ML Challenge 2026 - SageMaker Training

## Recommended Instance: ml.c5.9xlarge (36 vCPU, 72GB RAM)

**BEFORE STARTING - Upload files to S3 from your laptop:**
```bash
aws s3 mb s3://amazon-ml-2026-data
aws s3 cp 6ab10eb3b23ba_student_resource/student_resource/index.db s3://amazon-ml-2026-data/index.db
aws s3 cp index_test.db s3://amazon-ml-2026-data/index_test.db
aws s3 cp 6ab10eb3b23ba_student_resource/student_resource/dataset/ s3://amazon-ml-2026-data/dataset/ --recursive
```

Run cells ONE BY ONE in order.

In [ ]:
# Cell 1: Check Instance Specs
import os, multiprocessing, psutil

cpu_count = multiprocessing.cpu_count()
ram_gb = psutil.virtual_memory().total / 1e9
disk = psutil.disk_usage('/')

print(f'CPU cores  : {cpu_count}')
print(f'Total RAM  : {ram_gb:.1f} GB')
print(f'Available  : {psutil.virtual_memory().available / 1e9:.1f} GB')
print(f'Disk free  : {disk.free / 1e9:.0f} GB')
print()
NUM_WORKERS = min(cpu_count - 2, 68)
print(f'Recommended workers: {NUM_WORKERS}')
print('OK' if disk.free > 30e9 else 'WARNING: low disk space!')

In [ ]:
# Cell 2: Install Dependencies
!pip install lightgbm catboost xgboost pyarrow psutil --quiet
print('Dependencies installed')

In [ ]:
# Cell 3: Clone Repository
import os

REPO_URL = 'https://github.com/fallingbody/Amazon-ML-Hackathon-2026.git'
WORK_DIR = '/home/ec2-user/SageMaker/ml'

if os.path.exists(WORK_DIR):
    print('Repo exists, pulling latest...')
    os.system(f'cd {WORK_DIR} && git pull origin main')
else:
    print('Cloning...')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# Cell 4: Download Dataset + SQLite Indexes from S3
# S3 to SageMaker is very fast (~500 MB/s internally)
import subprocess, time

S3_BUCKET = 'amazon-ml-2026-data'  # Change if needed
WORK_DIR  = '/home/ec2-user/SageMaker/ml'
BASE = f'{WORK_DIR}/6ab10eb3b23ba_student_resource/student_resource'

def dl(s3_key, local, desc):
    os.makedirs(os.path.dirname(local), exist_ok=True)
    if os.path.exists(local):
        print(f'SKIP {desc} (already exists {os.path.getsize(local)/1e9:.1f}GB)')
        return
    print(f'Downloading {desc}...')
    t = time.time()
    r = subprocess.run(['aws','s3','cp',f's3://{S3_BUCKET}/{s3_key}',local], capture_output=True, text=True)
    if r.returncode == 0:
        print(f'  Done {os.path.getsize(local)/1e9:.1f}GB in {time.time()-t:.0f}s')
    else:
        print(f'  FAILED: {r.stderr}')

for split in ['train','test']:
    os.makedirs(f'{BASE}/dataset/{split}', exist_ok=True)

dl('index.db',      f'{BASE}/index.db',                              'Train SQLite Index (4.0GB)')
dl('index_test.db', f'{WORK_DIR}/index_test.db',                     'Test SQLite Index (4.3GB)')
dl('dataset/train/train_source1.tsv',      f'{BASE}/dataset/train/train_source1.tsv',      'train_source1 (201MB)')
dl('dataset/train/train_ground_truth.tsv', f'{BASE}/dataset/train/train_ground_truth.tsv', 'train_ground_truth (122MB)')
dl('dataset/train/train_source2.tsv',      f'{BASE}/dataset/train/train_source2.tsv',      'train_source2 (467MB)')
dl('dataset/train/train_source3.tsv',      f'{BASE}/dataset/train/train_source3.tsv',      'train_source3 (481MB)')
dl('dataset/test/test_source1.tsv',        f'{BASE}/dataset/test/test_source1.tsv',        'test_source1 (167MB)')
dl('dataset/test/test_source2.tsv',        f'{BASE}/dataset/test/test_source2.tsv',        'test_source2 (486MB)')
dl('dataset/test/test_source3.tsv',        f'{BASE}/dataset/test/test_source3.tsv',        'test_source3 (483MB)')

print('All downloads done!')

In [ ]:
# Cell 5: Verify All Files Present
import os

WORK_DIR = '/home/ec2-user/SageMaker/ml'
BASE = f'{WORK_DIR}/6ab10eb3b23ba_student_resource/student_resource'

required = [
    f'{BASE}/index.db',
    f'{WORK_DIR}/index_test.db',
    f'{BASE}/dataset/train/train_source1.tsv',
    f'{BASE}/dataset/train/train_ground_truth.tsv',
    f'{BASE}/dataset/train/train_source2.tsv',
    f'{BASE}/dataset/train/train_source3.tsv',
    f'{BASE}/dataset/test/test_source1.tsv',
    f'{BASE}/dataset/test/test_source2.tsv',
    f'{BASE}/dataset/test/test_source3.tsv',
]

all_ok = True
for f in required:
    exists = os.path.exists(f)
    tag = 'OK' if exists else 'MISSING'
    sz = f'{os.path.getsize(f)/1e9:.2f}GB' if exists else ''
    print(f'{tag:7} {os.path.basename(f):40} {sz}')
    if not exists: all_ok = False

print()
print('Ready to train!' if all_ok else 'Files missing - re-run Cell 4')

In [ ]:
# Cell 6: Set Training Configuration
import multiprocessing

cpu_count    = multiprocessing.cpu_count()
NUM_WORKERS  = min(cpu_count - 2, 30)
CHUNK_SIZE   = 200000
NUM_CHUNKS   = 8        # 8x200k = 1.6M records = 70% of dataset
CHUNK_OFFSET = 0
MAX_CANDS    = 80
VAL_SIZE     = 5000     # from 30% holdout tail
TREES_CHUNK  = 50       # total = 400 trees
WORK_DIR     = '/home/ec2-user/SageMaker/ml'

print('=== CONFIGURATION ===')
print(f'Workers         : {NUM_WORKERS} / {cpu_count} CPUs')
print(f'Chunks          : {NUM_CHUNKS} x {CHUNK_SIZE:,} = {NUM_CHUNKS*CHUNK_SIZE:,} records (70%)')
print(f'Max candidates  : {MAX_CANDS}')
print(f'Val size        : {VAL_SIZE:,}')
print(f'Total trees     : {NUM_CHUNKS*TREES_CHUNK}')

In [ ]:
# Cell 7: RUN TRAINING (Main Cell - takes 2-4 hours)
# Model is saved after EVERY chunk. Safe to stop between chunks.
# Watch for '>>> COMPETITION MACRO F0.5: X.XXXX <<<' after each chunk
import subprocess, sys, os

os.chdir(WORK_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = os.path.join(WORK_DIR, 'code', 'src')
env['OMP_NUM_THREADS'] = '1'
env['OPENBLAS_NUM_THREADS'] = '1'
env['MKL_NUM_THREADS'] = '1'

cmd = [
    sys.executable, 'code/src/main.py',
    '--split',           'train',
    '--chunk-size',      str(CHUNK_SIZE),
    '--num-chunks',      str(NUM_CHUNKS),
    '--chunk-offset',    str(CHUNK_OFFSET),
    '--max-candidates',  str(MAX_CANDS),
    '--val-size',        str(VAL_SIZE),
    '--num-workers',     str(NUM_WORKERS),
    '--trees-per-chunk', str(TREES_CHUNK),
]

print('Command:', ' '.join(cmd))
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'Exit code: {process.returncode}')
print('TRAINING COMPLETE' if process.returncode == 0 else 'TRAINING FAILED - check output')

In [ ]:
# Cell 8: RESUME (if Cell 7 was interrupted mid-way)
# Change COMPLETED_CHUNKS to how many chunks already finished
import subprocess, sys, os

COMPLETED_CHUNKS = 4   # <-- CHANGE THIS

remaining     = NUM_CHUNKS - COMPLETED_CHUNKS
resume_offset = COMPLETED_CHUNKS * CHUNK_SIZE
print(f'Resuming from chunk {COMPLETED_CHUNKS+1}, offset={resume_offset:,}, remaining={remaining}')

env = os.environ.copy()
env['PYTHONPATH'] = os.path.join(WORK_DIR, 'code', 'src')
env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable, 'code/src/main.py',
    '--split',           'train',
    '--chunk-size',      str(CHUNK_SIZE),
    '--num-chunks',      str(remaining),
    '--chunk-offset',    str(resume_offset),
    '--max-candidates',  str(MAX_CANDS),
    '--val-size',        str(VAL_SIZE),
    '--num-workers',     str(NUM_WORKERS),
    '--trees-per-chunk', str(TREES_CHUNK),
    '--continue-training',
]
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print(f'Exit: {process.returncode}')

In [ ]:
# Cell 9: RUN TEST INFERENCE (30-45 min)
import subprocess, sys, os

os.chdir(WORK_DIR)
env = os.environ.copy()
env['PYTHONPATH'] = os.path.join(WORK_DIR, 'code', 'src')
env['OMP_NUM_THREADS'] = '1'

cmd = [
    sys.executable, 'code/src/main.py',
    '--split',          'test',
    '--sample-size',    '0',
    '--max-candidates', str(MAX_CANDS),
    '--num-workers',    str(NUM_WORKERS),
]

print('Running test inference...')
process = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()
print('INFERENCE COMPLETE' if process.returncode == 0 else 'INFERENCE FAILED')

In [ ]:
# Cell 10: Validate + Package submission.zip
import os, subprocess

os.chdir(WORK_DIR)
MATCH = 'output/test/matching_results.tsv'
CANDS = 'output/test/candidate_pairs.tsv'
TEST_DIR = '6ab10eb3b23ba_student_resource/student_resource/dataset/test'

for f in [MATCH, CANDS]:
    ok = os.path.exists(f)
    sz = f'{os.path.getsize(f)/1e6:.1f}MB' if ok else 'MISSING'
    print(f'{"OK" if ok else "MISSING"}: {f} ({sz})')

print('\nValidating...')
r = subprocess.run(['python3','code/src/validate_submission.py','--matching',MATCH,'--candidate',CANDS,'--test-dir',TEST_DIR], capture_output=True, text=True)
print(r.stdout)
if r.stderr: print(r.stderr)

os.system(f'zip -j submission.zip {MATCH} {CANDS}')
print(f'submission.zip: {os.path.getsize("submission.zip")/1e6:.1f}MB')

In [ ]:
# Cell 11: Upload Results to S3 (then download to laptop)
import subprocess

S3_BUCKET = 'amazon-ml-2026-data'

uploads = [
    (f'{WORK_DIR}/submission.zip',                   f's3://{S3_BUCKET}/results/submission.zip'),
    (f'{WORK_DIR}/output/train/lgb_model.pkl',        f's3://{S3_BUCKET}/results/lgb_model.pkl'),
    (f'{WORK_DIR}/output/test/matching_results.tsv',  f's3://{S3_BUCKET}/results/matching_results.tsv'),
]

for local, s3 in uploads:
    r = subprocess.run(['aws','s3','cp',local,s3], capture_output=True, text=True)
    print(f'OK: {s3}' if r.returncode==0 else f'FAILED: {r.stderr.strip()}')

print(f'\nDownload to laptop:')
print(f'  aws s3 cp s3://{S3_BUCKET}/results/submission.zip ./')
print()
print('STOP YOUR SAGEMAKER INSTANCE NOW TO AVOID CHARGES!')